# Factor 示例

读取同目录的 `factor.json` 配置，调用 `analyze_factors`
在 DolphinDB 中完成因子预处理、IC 和分组收益分析，并按需下载结果。

## Goal

1. 从 `factor.json` 加载数据集查询与分析参数。
2. 使用内置预处理（MAD 去极值、标准化、市值与行业中性化）。
3. 下载预处理因子表、IC / Rank IC 时间序列和市值加权分组收益。
4. 使用 `with` 管理结果持有的 DolphinDB session。

## Setup

在项目根目录运行 `uv run jupyter lab`。DolphinDB 连接参数从项目的
`.env` 或 `DOLPHIN_HOST`、`DOLPHIN_PORT`、`DOLPHIN_USERNAME`、
`DOLPHIN_PASSWORD` 环境变量读取。

DolphinDB 中需要已经存在 CoreData 统一因子表及示例日期范围的数据，
内置预处理还需要可用的股票行业元数据。

In [1]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
load_dotenv(project_root / ".env")
load_dotenv(project_root.parent / ".env")

from core import analyze_factors
from core.utils import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

2

## Steps

### 1. 加载配置

`factor.json` 与 `manage.py factor` 命令共用同一套字段；notebook 只取
分析参数，忽略其中的 `output_dir`。收益率和市值列会自动补入
`dataset_query.factors`。

In [2]:
config_path = project_root / "examples" / "factor.json"
config = json.loads(config_path.read_text(encoding="utf-8"))

run_arguments = {
    key: value
    for key, value in config.items()
    if key != "output_dir"
}

print("factor_columns:", run_arguments["factor_columns"])
print("return_columns:", run_arguments["return_columns"])
print("n_groups:", run_arguments["n_groups"])
print("preprocess:", run_arguments["preprocess"])
print("industry_level:", run_arguments["industry_level"])

factor_columns: ['close', 'close_hfq']
return_columns: ['pct_chg', 'pct_chg_hfq']
n_groups: 5
preprocess: True
industry_level: sector


### 2. 执行分析并按需下载

`analyze_factors` 返回 `FactorAnalysisResult`。访问 `processed_data`、
`information_coefficients` 或 `all_group_returns` 时才从 session 下载；
退出 `with` 后 session 自动关闭。

In [3]:
with analyze_factors(**run_arguments) as factor_result:
    processed_data = factor_result.processed_data
    ic_tables = factor_result.information_coefficients
    group_return_tables = factor_result.all_group_returns

    print("session type:", type(factor_result.session).__name__)
    display(processed_data.head(10))
    for factor, ic in ic_tables.items():
        print(f"IC / Rank IC: {factor}")
        display(ic.head(10))
    for factor, group_returns in group_return_tables.items():
        print(f"分组收益: {factor}")
        display(group_returns.head(10))

print("session closed:", factor_result.closed)

2026-08-01 11:42:38.972 | INFO     | core.database.session:create_session:43 - DolphinDB: 1.13.198.44:8848


2026-08-01 11:42:39.083 | INFO     | core.apps.query.api:build_query_table:65 - session.run: 加载 query 模块


2026-08-01 11:42:39.114 | INFO     | core.database.session:has_session_variable:37 - session.run: 检查变量 coreFactorSourceData 是否存在


2026-08-01 11:42:39.119 | INFO     | core.apps.query.api:build_query_table:69 - session.run: 查询基础因子表 coreFactorSourceData


2026-08-01 11:42:39.150 | INFO     | core.apps.query.api:build_query_table:119 - session.run: 整理基础因子表 coreFactorSourceData


...

分组收益: close


,time,pct_chg_group0,pct_chg_group1,pct_chg_group2,pct_chg_group3,pct_chg_group4,pct_chg_hfq_group0,pct_chg_hfq_group1,pct_chg_hfq_group2,pct_chg_hfq_group3,pct_chg_hfq_group4
0,2025-03-03,-0.388534,-0.201973,-0.524239,-1.216101,-0.514496,-0.397886,-0.200730,-0.516756,-1.217031,-0.516319
1,2025-03-04,-0.733811,-1.618719,-0.012361,-0.885760,-0.167013,-0.700750,-1.625683,-0.011165,-0.887318,-0.166324
2,2025-03-05,1.129267,0.359960,0.420819,-0.271020,1.240344,1.107492,0.357584,0.414980,-0.269504,1.241320
3,2025-03-06,-0.445783,1.621891,0.959106,2.619339,0.287815,-0.421843,1.618616,0.960477,2.620074,0.290422
4,2025-03-07,-0.182723,-0.005823,0.264323,0.581635,0.613072,-0.218655,0.010638,0.265545,0.583032,0.612075


分组收益: close_hfq


,time,pct_chg_group0,pct_chg_group1,pct_chg_group2,pct_chg_group3,pct_chg_group4,pct_chg_hfq_group0,pct_chg_hfq_group1,pct_chg_hfq_group2,pct_chg_hfq_group3,pct_chg_hfq_group4
0,2025-03-03,-1.052091,-0.340544,-0.512922,-0.156478,-0.470277,-1.057865,-0.346946,-0.506666,-0.156549,-0.471795
1,2025-03-04,-0.700840,-0.082665,-0.012361,-1.849522,-0.648448,-0.672502,-0.084369,-0.011165,-1.859181,-0.647792
2,2025-03-05,0.608396,0.467681,0.861541,0.366821,0.754759,0.589607,0.465053,0.856584,0.363530,0.756951
3,2025-03-06,-0.196480,1.215617,1.170139,1.473960,1.359813,-0.177975,1.232111,1.169817,1.469219,1.361205
4,2025-03-07,0.170138,-0.975405,0.025817,-0.939359,1.021947,0.139341,-0.976974,0.031670,-0.913309,1.022418


session closed: True


## Checks

In [4]:
factor = run_arguments["factor_columns"][0]

assert factor_result.closed
assert not processed_data.empty
assert set(ic_tables) == set(run_arguments["factor_columns"])
assert set(group_return_tables) == set(run_arguments["factor_columns"])
assert f"{factor}_group" in processed_data.columns
assert {"time", "code", factor}.issubset(processed_data.columns)

print("Factor example checks passed.")

Factor example checks passed.


## Next Steps

- 调整 `factor.json` 中的 `codes`、日期范围和 `n_groups`。
- 在 `factor_columns` 中加入多个因子，或使用 `dataset_query.derivatives`
  构造衍生因子。
- 设置 `preprocess=false` 时，需要在 `dataset_query` 中自行输出
  `{factor}_group` 分组列。
- 使用 `factor_result.information_coefficient(factor)` 或
  `factor_result.group_returns(factor)` 下载单个因子的结果。